# Lab 03 — Combining Ideas
## Linear combinations, span, mixtures, and high-dimensional recipes

This lab is designed to be more than quick practice. It is a guided computational story.

In Chapter 3, we learn that vectors can be combined by scaling and adding. This simple operation creates one of the central ideas of linear algebra: **span**.

In this lab, you will:

1. Compute linear combinations by hand and with Python.
2. Visualize linear combinations as movement recipes.
3. Explore span in the plane.
4. Study convex combinations as mixtures.
5. Use vectors to blend images and signals.
6. Investigate high-dimensional linear combinations.
7. See how matrix-vector multiplication is a recipe for combining columns.

## 0. Setup

Run this cell first. We will use only standard scientific Python libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1. Linear combinations as recipes

A linear combination has the form

$$
c_1v_1+c_2v_2+\cdots+c_kv_k.
$$

The vectors $v_i$ are building blocks. The coefficients $c_i$ are recipe amounts.

In [ ]:
v1 = np.array([2, 1])
v2 = np.array([1, 3])

c1 = 2
c2 = 3

result = c1*v1 + c2*v2
result

### Student task 1

Change the coefficients `c1` and `c2` above. Try positive, negative, and fractional values.

Answer in words:

- What does a negative coefficient do geometrically?
- What does a coefficient between 0 and 1 do?
- What does a large coefficient do?

## 2. Drawing vector recipes

The next function draws two building-block vectors and their linear combination.

In [ ]:
def draw_linear_combination(v1, v2, c1, c2, title=None):
    v1 = np.array(v1, dtype=float)
    v2 = np.array(v2, dtype=float)
    r = c1*v1 + c2*v2

    plt.figure(figsize=(7, 7))
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)

    # Building blocks
    plt.quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1, label='v1')
    plt.quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1, label='v2')

    # Scaled versions
    plt.quiver(0, 0, (c1*v1)[0], (c1*v1)[1], angles='xy', scale_units='xy', scale=1, alpha=0.7, label='c1 v1')
    plt.quiver((c1*v1)[0], (c1*v1)[1], (c2*v2)[0], (c2*v2)[1], angles='xy', scale_units='xy', scale=1, alpha=0.7, label='c2 v2')

    # Result
    plt.quiver(0, 0, r[0], r[1], angles='xy', scale_units='xy', scale=1, linewidth=2, label='result')

    all_pts = np.vstack([[0,0], v1, v2, c1*v1, c2*v2, r])
    m = max(2, np.abs(all_pts).max() + 1)
    plt.xlim(-m, m)
    plt.ylim(-m, m)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.grid(True)
    plt.legend()
    plt.title(title or f'{c1} v1 + {c2} v2 = {r}')
    plt.show()


draw_linear_combination([2, 1], [1, 3], 2, 3)

### Student task 2

Use the function above to draw:

1. $v_1=(2,1)$, $v_2=(1,3)$, $c_1=1$, $c_2=-1$.
2. $v_1=(1,0)$, $v_2=(0,1)$, $c_1=4$, $c_2=2$.
3. $v_1=(2,1)$, $v_2=(4,2)$, $c_1=1$, $c_2=1$.

What is special about the third case?

In [ ]:
# Try your examples here
# draw_linear_combination(...)

## 3. Span of one vector

The span of one nonzero vector is a line through the origin.

$$
\operatorname{span}\{v\}=\{cv:c\in\mathbb{R}\}.
$$

In [ ]:
v = np.array([2, 1], dtype=float)
cs = np.linspace(-4, 4, 100)
points = np.array([c*v for c in cs])

plt.figure(figsize=(7, 7))
plt.plot(points[:,0], points[:,1])
plt.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1)
plt.scatter(points[::10,0], points[::10,1])
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.gca().set_aspect('equal', adjustable='box')
plt.grid(True)
plt.title('Span of one vector')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

### Reflection

Why must this line pass through the origin?

Write your answer here:

> ...

## 4. Span of two vectors in the plane

Two vectors in $\mathbb{R}^2$ can behave in two very different ways:

- If one is a scalar multiple of the other, their span is a line.
- If they point in genuinely different directions, their span is the whole plane.

The next code samples many linear combinations.

In [ ]:
def plot_span_samples(v1, v2, coefficient_range=(-3, 3), n=25, title=None):
    v1 = np.array(v1, dtype=float)
    v2 = np.array(v2, dtype=float)
    cs = np.linspace(coefficient_range[0], coefficient_range[1], n)
    pts = []
    for a in cs:
        for b in cs:
            pts.append(a*v1 + b*v2)
    pts = np.array(pts)

    plt.figure(figsize=(7, 7))
    plt.scatter(pts[:,0], pts[:,1], s=12, alpha=0.65)
    plt.quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1)
    plt.quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1)
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.grid(True)
    plt.title(title or 'Sampled linear combinations')
    plt.show()

plot_span_samples([1, 0], [0, 1], title='Two independent directions fill the plane')
plot_span_samples([2, 1], [4, 2], title='Two dependent directions still make only a line')

### Student task 3

Try several pairs of vectors. For each pair, predict whether the sampled points will form a line-like set or a plane-like cloud before you run the code.

In [ ]:
# Try your own vector pairs here
# plot_span_samples([?, ?], [?, ?])

## 5. Can we build the target?

A central question is:

> Can $b$ be written as $c_1v_1+c_2v_2+\cdots+c_kv_k$?

If yes, then $b$ is in the span of the building blocks.

When the building blocks are columns of a matrix $A$, this question becomes:

$$
Ac=b.
$$

In [ ]:
v1 = np.array([1, 2], dtype=float)
v2 = np.array([3, 1], dtype=float)
b = np.array([7, 8], dtype=float)

A = np.column_stack([v1, v2])
c = np.linalg.solve(A, b)

print('A =')
print(A)
print('coefficients =', c)
print('A @ c =', A @ c)
print('target b =', b)

### Student task 4

Try to build different target vectors from the same two building blocks. Then change the building blocks.

What happens when the two building blocks are multiples of each other?

In [ ]:
# Experiment here

## 6. Convex combinations: mixtures and averages

A convex combination is a linear combination where the coefficients are nonnegative and add to 1.

For two vectors, the convex combinations are

$$
(1-t)u+tv, \qquad 0\le t\le 1.
$$

These points form the line segment between $u$ and $v$.

In [ ]:
u = np.array([1, 2], dtype=float)
v = np.array([6, 4], dtype=float)
ts = np.linspace(0, 1, 11)
segment = np.array([(1-t)*u + t*v for t in ts])

plt.figure(figsize=(7, 6))
plt.plot(segment[:,0], segment[:,1], marker='o')
for t, p in zip(ts, segment):
    plt.text(p[0]+0.05, p[1]+0.05, f'{t:.1f}', fontsize=8)
plt.scatter([u[0], v[0]], [u[1], v[1]], s=100)
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.gca().set_aspect('equal', adjustable='box')
plt.grid(True)
plt.title('Convex combinations interpolate between two points')
plt.show()

## 7. Three-way mixtures: a triangle of possibilities

If three coefficients are nonnegative and add to 1, then the convex combinations of three points fill a triangle.

In [ ]:
p1 = np.array([0, 0], dtype=float)
p2 = np.array([5, 0], dtype=float)
p3 = np.array([2, 4], dtype=float)

pts = []
weights = []
for a in np.linspace(0, 1, 31):
    for b in np.linspace(0, 1-a, 31):
        c = 1 - a - b
        pts.append(a*p1 + b*p2 + c*p3)
        weights.append([a, b, c])
pts = np.array(pts)
weights = np.array(weights)

plt.figure(figsize=(7, 6))
plt.scatter(pts[:,0], pts[:,1], s=12, alpha=0.7)
plt.plot([p1[0], p2[0], p3[0], p1[0]], [p1[1], p2[1], p3[1], p1[1]])
plt.scatter([p1[0], p2[0], p3[0]], [p1[1], p2[1], p3[1]], s=100)
plt.text(p1[0], p1[1]-0.25, 'p1')
plt.text(p2[0], p2[1]-0.25, 'p2')
plt.text(p3[0], p3[1]+0.15, 'p3')
plt.gca().set_aspect('equal', adjustable='box')
plt.grid(True)
plt.title('Convex combinations of three points fill a triangle')
plt.show()

## 8. Signals as linear combinations

A complicated signal can be created by combining simple waves.
This is the doorway to Fourier analysis.

In [ ]:
t = np.linspace(0, 1, 600)
wave1 = np.sin(2*np.pi*2*t)
wave2 = np.sin(2*np.pi*5*t)
wave3 = np.sin(2*np.pi*11*t)

signal = 1.0*wave1 + 0.5*wave2 + 0.25*wave3

plt.figure(figsize=(10, 4))
plt.plot(t, signal, label='combined signal')
plt.plot(t, wave1, alpha=0.5, label='wave 1')
plt.plot(t, 0.5*wave2, alpha=0.5, label='0.5 wave 2')
plt.plot(t, 0.25*wave3, alpha=0.5, label='0.25 wave 3')
plt.legend()
plt.title('A signal built as a linear combination of waves')
plt.xlabel('time')
plt.grid(True)
plt.show()

### Student task 5

Change the frequencies and coefficients in the previous cell.

Questions:

- Which coefficient controls the largest visible pattern?
- What happens when you add a high-frequency wave with a small coefficient?

## 9. Images as high-dimensional vectors

Each image below is a $60\times 60$ grid. After flattening, each image is a vector in $\mathbb{R}^{3600}$.

We can still form linear combinations.

In [ ]:
n = 60
x = np.linspace(-1, 1, n)
X, Y = np.meshgrid(x, x)

img_left = np.exp(-14*((X+0.35)**2 + (Y+0.05)**2))
img_right = np.exp(-14*((X-0.35)**2 + (Y-0.05)**2))
img_ring = np.exp(-40*(np.sqrt(X**2+Y**2)-0.45)**2)

blend1 = 0.6*img_left + 0.4*img_right
blend2 = 0.25*img_left + 0.25*img_right + 0.5*img_ring

images = [img_left, img_right, img_ring, blend1, blend2]
titles = ['left blob', 'right blob', 'ring', '0.6 left + 0.4 right', 'three-way mixture']

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, im, title in zip(axes, images, titles):
    ax.imshow(im, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.show()

print('Each image vector has dimension:', n*n)

## 10. Matrix-vector multiplication as column combination

The expression $Ac$ is a linear combination of the columns of $A$.

If

$$
A=[v_1 \ v_2 \ \cdots \ v_k],
$$

then

$$
Ac=c_1v_1+c_2v_2+\cdots+c_kv_k.
$$

In [ ]:
A = np.array([
    [1, 3, 0],
    [2, 1, 5],
    [0, 4, 2],
    [1, 0, 1]
], dtype=float)

c = np.array([2, -1, 0.5])

print('A @ c =')
print(A @ c)

manual = c[0]*A[:,0] + c[1]*A[:,1] + c[2]*A[:,2]
print('Manual column combination =')
print(manual)

## 11. High-dimensional span experiment

In high dimensions, we cannot draw the span directly. But we can ask numerical questions.

Suppose we have $k$ random building blocks in $\mathbb{R}^{100}$. We create many random recipes and observe the resulting vectors.

Even though the vectors live in 100 dimensions, if we only use $k$ building blocks, all results lie in at most a $k$-dimensional subspace.

In [ ]:
np.random.seed(4)
d = 100   # ambient dimension
k = 3     # number of building blocks
m = 500   # number of random recipes

V = np.random.randn(d, k)
C = np.random.randn(k, m)
X = V @ C  # each column is a high-dimensional result

print('V shape:', V.shape)
print('C shape:', C.shape)
print('X shape:', X.shape)

# Numerical rank estimates the dimension of the span sampled
rank_X = np.linalg.matrix_rank(X)
rank_V = np.linalg.matrix_rank(V)
print('rank of building blocks V:', rank_V)
print('rank of generated data X:', rank_X)

### Reflection

Even though `X` contains 500 vectors in $\mathbb{R}^{100}$, its rank is small.
Why?

Write your answer here:

> ...

## 12. A small PCA preview

The generated high-dimensional data from the previous section actually lives in a low-dimensional span. We can use singular value decomposition to see this.

You do not need to understand SVD yet. For now, just notice that only a few directions matter.

In [ ]:
# Center the data columns
X_centered = X - X.mean(axis=1, keepdims=True)
U, S, VT = np.linalg.svd(X_centered, full_matrices=False)

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, 16), S[:15], marker='o')
plt.title('Only a few singular values are large')
plt.xlabel('index')
plt.ylabel('singular value')
plt.grid(True)
plt.show()

## 13. Final mini-project

Create your own example of linear combinations in one of the following settings:

1. **Movement:** Combine movement vectors to create a path.
2. **Food/product features:** Combine feature vectors to create a new product profile.
3. **Signal:** Combine waves to make a more complicated signal.
4. **Image:** Combine synthetic images.
5. **High-dimensional data:** Generate random building blocks and random recipes.

Your mini-project should include:

- a clear story,
- the vectors being combined,
- the coefficients,
- at least one visualization,
- a short explanation of what the span means in your example.

In [ ]:
# Final mini-project workspace

## 14. Summary

In this lab, you practiced the central idea of Chapter 3:

$$
c_1v_1+c_2v_2+\cdots+c_kv_k.
$$

A linear combination is a recipe. The span is everything the recipe system can create.

This idea will return again and again: in matrices, solving equations, projections, data analysis, PCA, image compression, and machine learning.